In [1]:
import matplotlib.pyplot as pyplot
import pandas as pd
import geopandas as gpd
import numpy as np
from google.colab import drive

In [2]:
drive.mount('/content/drive/')

Mounted at /content/drive/


In [4]:
static = pd.read_parquet("/content/drive/MyDrive/Nino/intermediate_merged_data/static_all.parquet")

In [5]:
static

,GEOID,hand_vulnerability,dist_coast_m,pct_no_vehicle
0,29510,0.397792,923183.623826,0.202684
1,32510,0.224161,263109.717316,0.058145
2,51580,0.209800,263425.100864,0.079286
3,51530,0.195965,210610.624338,0.046081
4,51678,0.202080,216861.155346,0.095262
...,...,...,...,...
3121,44003,0.295109,20686.443028,0.058925
3122,09007,0.228851,0.000000,0.054371
3123,09013,0.181630,36268.596654,0.044974
3124,09003,0.297104,35719.921820,0.105663


In [9]:
# import the counties and get their centroids for lat/lon
counties = gpd.read_file("/content/drive/MyDrive/Nino/raw_data/county_boundaries/tl_2018_us_county.shp").to_crs(epsg=5070) # equal-area projection
counties["lon"] = counties.geometry.centroid.to_crs(epsg=4326).x # then convert back to lat/lon degrees and get centroid
counties["lat"] = counties.geometry.centroid.to_crs(epsg=4326).y

region = counties[["GEOID", "lat", "lon"]]
region["GEOID"] = region["GEOID"].astype(str).str.zfill(5)
region

/tmp/ipykernel_1396/3879678691.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  region["GEOID"] = region["GEOID"].astype(str).str.zfill(5)


,GEOID,lat,lon
0,31039,41.916311,-96.787400
1,53069,46.291139,-123.433359
2,35011,34.342297,-104.412089
3,31109,40.783894,-96.687757
4,31129,40.176298,-98.047185
...,...,...,...
3228,13123,34.691158,-84.455615
3229,27135,48.775210,-95.810863
3230,28089,32.634711,-90.034036
3231,48227,32.306107,-101.435512


In [11]:
updated_static_all = static.merge(region, on="GEOID", how="left")
updated_static_all.isna().sum()

,0
GEOID,0
hand_vulnerability,0
dist_coast_m,0
pct_no_vehicle,0
lat,0
lon,0


In [12]:
region.to_parquet("/content/drive/MyDrive/Nino/cleaned_data/region_static.parquet")
updated_static_all.to_parquet("/content/drive/MyDrive/Nino/intermediate_merged_data/updated_static_all.parquet")

updated_static_all

,GEOID,hand_vulnerability,dist_coast_m,pct_no_vehicle,lat,lon
0,29510,0.397792,923183.623826,0.202684,38.635746,-90.245190
1,32510,0.224161,263109.717316,0.058145,39.151182,-119.747419
2,51580,0.209800,263425.100864,0.079286,37.778613,-79.986757
3,51530,0.195965,210610.624338,0.046081,37.731925,-79.356833
4,51678,0.202080,216861.155346,0.095262,37.782530,-79.443895
...,...,...,...,...,...,...
3121,44003,0.295109,20686.443028,0.058925,41.673390,-71.579071
3122,09007,0.228851,0.000000,0.054371,41.435207,-72.522944
3123,09013,0.181630,36268.596654,0.044974,41.854894,-72.336536
3124,09003,0.297104,35719.921820,0.105663,41.806252,-72.732804
